In [5]:
import numpy.random as rnd
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import scipy as sp
from scipy.stats import norm
import pandas_datareader as pd_data
from pandas_datareader import data as pdr
from datetime import datetime
from scipy.interpolate import interp1d 
# from scipy import arange, array, exp
from scipy.optimize import brentq
from scipy.interpolate import griddata
import datacleaning as dc
from opengreeks import black76
import os

In [2]:
def Black76_OptionPrice(St,tau,F,K,dir,sigma):
    """
    St: current asset price (at time t)
    tau: time to expiry of the option
    F: forward price
    K: strike
    sigma: volatility
    ***all of the above can be both vectors and scalars***
    returns CALL PRICE(s) using the Black-76 formula
    """
    efact = F/St
    #delta of the option
    d1 = (np.log(F/K)+tau*0.5*sigma*sigma)/(sigma*np.sqrt(tau))
    d2 = d1-sigma*np.sqrt(tau)
    price = efact*(F*norm.cdf(d1)-K*norm.cdf(d2))
    if dir == 'CE':
        price = efact*(F*norm.cdf(d1)-K*norm.cdf(d2))
    elif dir == 'PE': 
        price = efact*(-F*norm.cdf(-d1)+K*norm.cdf(-d2))
    return price

def BS_OptionPrice(St,tau,r,K,dir,sigma):
    """
    St: current asset price (at time t)
    tau: time to expiry of the option
    r: risk-free interest rate
    K: strike
    sigma: volatility
    ***all of the above can be both vectors and scalars***
    returns CALL PRICE(s) using the Black-Scholes formula
    """
    #delta
    d1 = (np.log(St/K)+tau*(r+0.5*sigma*sigma))/(sigma*np.sqrt(tau))
    d2 = d1-sigma*np.sqrt(tau)
    if dir == 'CE':
        price = St*norm.cdf(d1)-K*norm.cdf(d2)*np.exp(-r*tau)
    elif dir == 'PE': 
        price = K*norm.cdf(-d2)*np.exp(-r*tau) - St*norm.cdf(-d1)

    return price

#Converting from K,T to m,tau
def K_T_to_mu_tau(K,T,St,t):
    return K/St,T-t
def mu_tau_to_K_T(mu,tau,St,t):
    return mu*St,tau+t

In [ ]:
# %load_ext autoreload
# %autoreload 2

In [3]:
df_niftyoptions= pd.read_csv('data/niftyoptions_df.csv')
df_niftyspot = pd.read_csv('data/nifty_close.csv')

niftyoptions_df, niftyspot_df, spot, observation_dates = dc.optiondata_transform(df_niftyoptions, df_niftyspot)
datapath = 'data/datapath.csv'

def create_data(
    observation_dates,
    niftyoptions_df,
    niftyspot_df,
    m_grid=None,
    tau_days=None,
    datapath=datapath
):
    """
    datapath: Option-level data with calculated implied volatility.
    Parameters
    ----------
    observation_dates : iterable Trading observation_dates to process.
    niftyoptions_df : pd.DataFrame Preprocessed NIFTY option data.
    niftyspot_df : pd.DataFrame NIFTY spot data.
    m_grid : array-like Fixed moneyness grid.
    tau_days : array-like Fixed maturity grid in calendar days.
    datapath : str Output path for option-level data.
    surfacepath : str Output path for flattened surfaces.
    """

    if m_grid is None:
        m_grid = np.linspace(0.6, 1.4, 10)

    if tau_days is None:
        tau_days = np.array([7, 14, 30, 60,90, 180, 270, 365])

    tau_grid = np.asarray(tau_days) / 365.0

    data_list = []
    surface_list = []

    for date in observation_dates:

        subset_df, r1 = dc.data_day(date,niftyoptions_df,niftyspot_df)
        call_df = subset_df[subset_df["option_type"] == "CE"].copy()
        put_df = subset_df[subset_df["option_type"] == "PE"].copy()

        def solve_iv(row):
            F = row["spot"]
            St = row['spot']
            tau = row["ttm"]
            K = row["strike"]
            market_price = row["mid_price"]
            dir = row["option_type"]
            r = row['r']

            if (pd.isna(St) or pd.isna(tau) or pd.isna(K) or pd.isna(market_price)):
                return np.nan

            if St <= 0 or tau <= 0 or market_price <= 0:
                return np.nan

            def objective(sigma):
                model_price = Black76_OptionPrice(St,tau,F,K,dir,sigma)
                return model_price - market_price

            try:
                return brentq(objective, 1e-6, 5.0)
            except ValueError:
                return np.nan

        call_df["iv"] = call_df.apply( solve_iv, axis=1)
        call_df = call_df[call_df["iv"].notna()].copy()
        call_df = call_df[(call_df["iv"] > 0) & (call_df["iv"] < 5)].copy()
        data_list.append(call_df)

        put_df["iv"] = put_df.apply( solve_iv, axis=1)
        put_df = put_df[put_df["iv"].notna()].copy()
        put_df = put_df[(put_df["iv"] > 0) & (put_df["iv"] < 5)].copy()
        data_list.append(put_df)

    if len(data_list) > 0:
        data_output = pd.concat(data_list,ignore_index=True)
    else:
        data_output = pd.DataFrame()

    data_output.to_csv(datapath,index=False)
    print(f"datapath saved to: {datapath}")
    print(f"datapath shape: {data_output.shape}")

    return data_output

In [4]:
# subset_df, r = dc.data_day(observation_dates[0],niftyoptions_df,niftyspot_df)

In [ ]:
if os.path.exists('data/nifty_df.parquet') and False:
    nifty_df = pd.read_parquet('data/nifty_df.parquet')    
else:
    nifty_df = create_data(observation_dates,niftyoptions_df,niftyspot_df)
    nifty_df.to_parquet('data/nifty_df.parquet', index = False)

In [ ]:
greek_functions = {
    "delta": black76.delta,
    "gamma": black76.gamma,
    "vega": black76.vega,
    "theta": black76.theta,
    "rho": black76.rho,
    "vanna": black76.vanna,
    "charm": black76.charm,
    "vomma": black76.vomma,
    "speed": black76.speed,
    "zomma": black76.zomma,
    "color": black76.color,
    "veta": black76.veta,
    "ultima": black76.ultima,
    "dual_delta": black76.dual_delta,
    "dual_gamma": black76.dual_gamma,
}


def calculate_greeks(row):
    F = float(row["spot"])
    K = float(row["strike"])
    t = float(row["ttm"])
    RATE = float(row["r"])
    price = float(row["close"])

    flag = {"CE": "c", "PE": "p"}.get(str(row["option_type"]).upper())

    result = {
        "implied_volatility": np.nan,
        "model_price": np.nan,
        "greek_error": None,
    }
    
    if flag is None or not all(
        np.isfinite(x) for x in [F, K, t, price]
    ) or F <= 0 or K <= 0 or t <= 0 or price <= 0:
        result["greek_error"] = "Invalid input"
        return pd.Series(result)

    try:
        iv = black76.implied_volatility(price, F, K, RATE, t, flag)
        if not np.isfinite(iv) or iv <= 0:
            raise ValueError("Invalid implied volatility")

        result["implied_volatility"] = iv
        result["model_price"] = black76.black(flag, F, K, t, RATE, iv)

        for name, func in greek_functions.items():
            result[name] = func(flag, F, K, t, RATE, iv)

    except Exception as exc:
        result["greek_error"] = str(exc)
    return pd.Series(result)

In [ ]:
if os.path.exists('data/df_with_greeks.parquet') and False:
    df_with_greeks = pd.read_parquet('data/df_with_greeks.parquet')    
else:
    greek_columns = nifty_df.apply(calculate_greeks, axis=1)

    df_with_greeks = pd.concat(
        [nifty_df.reset_index(drop=True),
        greek_columns.reset_index(drop=True)],
        axis=1,
    )
    df_with_greeks.to_parquet('data/df_with_greeks.parquet', index = False)

In [ ]:
nifty_df[(nifty_df['date'] == '2025-01-01')]

In [ ]:
nifty_df[(nifty_df['date'] == '2025-01-01')&(nifty_df['days_to_exp']==15)& (nifty_df['option_type'] == 'CE')].sort_values('strike')

In [ ]:
df_with_greeks
df_with_greeks[(df_with_greeks['date'] == '2025-01-01')&(df_with_greeks['days_to_exp']==15)& (df_with_greeks['option_type'] == 'CE')][list(nifty_df.columns) + ['delta']].sort_values('strike')


In [ ]:
dts = df_with_greeks['date'].unique()
numeric_cols = ["strike", "close", "ttm", "days_to_exp","delta", "gamma", "vega", "theta", "rho","vanna",
                 "charm", "vomma", "speed", "zomma","color", "veta", "ultima", "dual_delta", "dual_gamma"]
day_to_exp = np.sort(df_with_greeks['days_to_exp'].unique())

In [ ]:
df = df_with_greeks[(df_with_greeks['date'] == dts[0])] 
plt.figure(figsize=(10, 6))

for dt in df['days_to_exp'].unique():
    for opt in ['CE', 'PE']:
        temp = df[(df['option_type'] == opt) & (df['days_to_exp'] == dt)].sort_values('strike')

        plt.plot(temp['strike'],temp['delta'],marker='o',label=opt)

plt.axhline(0, linestyle='--', linewidth=0.8)
plt.xlabel('Strike')
plt.ylabel('Delta')
plt.title(f'Option Delta vs Strike — {dts[0]}')
# plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

data = df_with_greeks.copy()
for col in ["date_dt", "exdate_dt", "expiry"]:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors="coerce")

numeric_cols = [
    "strike", "close", "ttm", "days_to_exp",
    "delta", "gamma", "vega", "theta", "rho",
    "vanna", "charm", "vomma", "speed", "zomma",
    "color", "veta", "ultima", "dual_delta", "dual_gamma",
]
for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

# Find the available sensitivities
greeks = [c for c in numeric_cols if c in data.columns
          and c not in ["strike", "close", "ttm", "days_to_exp"]]

# Derive moneyness only if a forward/spot column is available
underlying_col = next(
    (c for c in ["forward", "futures_price", "spot", "underlying_price"]
     if c in data.columns),
    None
)

if underlying_col:
    data["moneyness"] = data["strike"] / data[underlying_col]
    data["log_moneyness"] = np.log(data["moneyness"])

greek_w = widgets.Dropdown(
    options=greeks, description="Greek:"
)
type_w = widgets.Dropdown(
    options=["All", "CE", "PE"], description="Option:"
)
plot_w = widgets.Dropdown(
    options=[
        ("Greek vs strike", "strike"),
        ("Greek vs expiry days", "days_to_exp"),
        ("Greek vs moneyness", "log_moneyness"),
        ("Strike-expiry heatmap", "heatmap"),
    ],
    description="Plot:"
)
date_w = widgets.Dropdown(
    options=["All"] + sorted(data["date"].dropna().astype(str).unique().tolist()),
    description="Date:"
)
out = widgets.Output()

def draw(greek, opt_type, plot_type, date):
    with out:
        clear_output(wait=True)
        d = data.copy()

        if opt_type != "All":
            d = d[d["option_type"] == opt_type]
        if date != "All":
            d = d[d["date"].astype(str) == date]

        if plot_type == "log_moneyness" and "log_moneyness" not in d:
            print("Add a historical spot or forward column to calculate moneyness.")
            return

        if plot_type == "heatmap":
            d = d.dropna(subset=["strike", "days_to_exp", greek])
            if d.empty:
                print("No valid data for this selection.")
                return

            z = d.pivot_table(
                index="days_to_exp",
                columns="strike",
                values=greek,
                aggfunc="median"
            ).sort_index()

            fig = px.imshow(
                z,
                aspect="auto",
                origin="lower",
                labels=dict(
                    x="Strike", y="Days to expiry", color=greek
                ),
                title=f"{greek} across strike and expiry"
            )
        else:
            d = d.dropna(subset=[plot_type, greek])
            if d.empty:
                print("No valid data for this selection.")
                return

            fig = px.scatter(
                d,
                x=plot_type,
                y=greek,
                color="option_type",
                hover_data=[
                    c for c in ["strike", "expiry", "close", "ttm"]
                    if c in d.columns
                ],
                title=f"{greek} vs {plot_type}",
                opacity=0.65,
            )

        fig.update_layout(template="plotly_white")
        fig.show()

def refresh(change=None):
    draw(greek_w.value, type_w.value, plot_w.value, date_w.value)

for w in [greek_w, type_w, plot_w, date_w]:
    w.observe(refresh, names="value")

display(widgets.HBox([greek_w, type_w]))
display(widgets.HBox([plot_w, date_w]))
display(out)
refresh()
